# Session 1: From Text to Signals

This notebook introduces four building blocks:
1. sentence embeddings and semantic similarity
2. a minimal fine-tuning example
3. embedding-based zero-shot classification
4. NLI-based zero-shot classification


## 1. Embeddings and Similarity

Embeddings map text into dense vectors. Once we have those vectors, we can compare texts, cluster them, and visualize semantic structure.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

from FewShotX import configure_notebook
_, _ = configure_notebook(theme="whitegrid", progress_bar="rich")

In [ ]:
# A tiny corpus with overlapping and contrasting meanings.
sentences = [
    "The military seized power in the capital.",
    "The army overthrew the civilian government.",
    "A peaceful election was held in the country.",
    "Inflation expectations remained stable.",
]

labels = [f"S{i+1}" for i in range(len(sentences))]

In [ ]:
# Embed the sentences with a lightweight sentence-transformer model.
model = SentenceTransformer("all-MiniLM-L6-v2")
Z = model.encode(sentences, normalize_embeddings=True)
print(f"Embedding shape: {Z.shape}")

In [ ]:
# Compare each sentence against the others with cosine similarity.
sim_matrix = cosine_similarity(Z)
df_sim = pd.DataFrame(sim_matrix, index=labels, columns=labels)

print("Cosine similarity matrix:")
df_sim.round(2)

In [ ]:
# Project the embeddings to two dimensions for a quick visual check.
pca = PCA(n_components=2)
Z_2d = pca.fit_transform(Z)

plt.figure(figsize=(8, 5))
plt.scatter(Z_2d[:, 0], Z_2d[:, 1])

for i, sentence in enumerate(sentences):
    short_text = sentence[:35] + "..."
    plt.text(Z_2d[i, 0] + 0.01, Z_2d[i, 1] + 0.01, f"{labels[i]}: {short_text}", fontsize=10)

plt.axhline(0, linewidth=0.5)
plt.axvline(0, linewidth=0.5)
plt.title("Sentence embeddings in 2D (PCA)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()

## 2. A Minimal Fine-Tuning Example

We now train a tiny supervised model on top of a transformer encoder. The goal is not performance, but to make the training loop and prediction flow concrete.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
from torch.utils.data import Dataset


class ToyDataset(Dataset):
    """Six short texts paired with three-dimensional target vectors."""

    def __init__(self):
        self.samples = [
            ("I love machine learning", [1, 0, 0]),
            ("I enjoy deep learning", [1, 0, 0]),
            ("Cats are cute", [0, 1, 0]),
            ("Dogs are loyal", [0, 1, 0]),
            ("Python is great for programming", [0, 0, 1]),
            ("I code in Python", [0, 0, 1]),
        ]
        self.tokenizer = AutoTokenizer.from_pretrained(
            "distilbert-base-uncased",
            clean_up_tokenization_spaces=False,
        )
        self.tokenizer.clean_up_tokenization_spaces = False

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        text, label = self.samples[idx]
        inputs = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=16,
            return_tensors="pt",
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.float),
        }

### 2.1 Model


In [ ]:
class SimpleEmbedder(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModel.from_pretrained("distilbert-base-uncased")
        self.embedding_layer = torch.nn.Linear(768, 3)
        self.loss_fn = torch.nn.KLDivLoss(reduction="batchmean")

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        logits = self.embedding_layer(cls_output)
        probs = torch.softmax(logits, dim=-1)

        if labels is not None:
            log_probs = torch.log_softmax(logits, dim=-1)
            loss = self.loss_fn(log_probs, labels)
            return {"loss": loss, "logits": logits, "probs": probs}

        return {"logits": logits, "probs": probs}

### 2.2 Training


In [ ]:
torch.manual_seed(42)

dataset = ToyDataset()
train_args = TrainingArguments(
    output_dir="./embeddings_model",
    per_device_train_batch_size=2,
    num_train_epochs=5,
    logging_dir="./logs",
    logging_steps=1,
    report_to="none",
    save_strategy="no",
    dataloader_pin_memory=False,
)

model = SimpleEmbedder()
trainer = Trainer(model=model, args=train_args, train_dataset=dataset)
trainer.train()

### 2.3 Inspect The Predicted Probabilities


In [ ]:
model.eval()
sample_text = "I love coding"

tokenizer = dataset.tokenizer
device = next(model.parameters()).device
inputs = tokenizer(sample_text, return_tensors="pt")
inputs = {name: tensor.to(device) for name, tensor in inputs.items()}

with torch.no_grad():
    output = model(**inputs)

print("Logits:")
print(output["logits"].detach().cpu().numpy())

print("\nPredicted probabilities:")
print(output["probs"].detach().cpu().numpy())

## 3. Embedding-Based Zero-Shot Classification

Instead of training a classifier, we can embed the text and the candidate labels in the same space and compare them directly.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt

model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# Example text and candidate labels.
text = "Opposition party gains ground ahead of national election."
labels = ["economics", "sports", "politics"]

embed_text = model.encode([text])
embed_labels = model.encode(labels)

In [ ]:
# Rescale cosine similarity to the [0, 1] range for easier reading.
sim_matrix = cosine_similarity(embed_text, embed_labels)
sim_matrix = (sim_matrix + 1) / 2
print("Similarity scores:", sim_matrix[0])

plt.figure(figsize=(6, 2))
plt.barh(labels, sim_matrix.flatten(), color="skyblue")
plt.xlabel("Probability-like score")
plt.title("Embedding-based zero-shot scores")
plt.xlim(0, 1)
plt.gca().invert_yaxis()
plt.grid(alpha=0.3, linestyle="--", axis="x")
plt.tight_layout()
plt.show()

## 4. Zero-Shot Classification With NLI

A Natural Language Inference model treats each label as a hypothesis and asks whether the input text entails it.


In [ ]:
import torch
import matplotlib.pyplot as plt

def ZeroShotNLI(tokenizer, model, text, candidate_labels, multi_label=False, plot=False):
    """
    Perform zero-shot classification using a Natural Language Inference (NLI) model.

    This function reformulates each candidate label as a hypothesis and computes the likelihood 
    that the input text (premise) entails each hypothesis. It supports both single-label 
    (multi-class) and multi-label classification modes.

    Parameters
    ----------
    tokenizer : transformers.PreTrainedTokenizer
        The tokenizer associated with the pre-trained NLI model.
    model : transformers.PreTrainedModel
        The NLI model used to predict entailment probabilities.
    text : str
        The input text to classify (used as the premise in the NLI formulation).
    candidate_labels : List[str]
        A list of labels, each of which will be turned into a natural language hypothesis.
    multi_label : bool, optional (default=False)
        - If False, assumes only one label is true (mutually exclusive labels).
        - If True, multiple labels can be simultaneously "true".

    Returns
    -------
    dict
        A dictionary with:
            - "sequence": The input text.
            - "labels": The original list of candidate labels.
            - "scores": Probabilities corresponding to each label.
    """
    # Template to transform each label into a natural language hypothesis
    hypothesis_template = "This text is about {}."
    hypotheses = [hypothesis_template.format(label) for label in candidate_labels]

    # Identify the position of "entailment" in the model's output logits
    entailment_id = model.config.label2id.get("entailment", 2)

    # Tokenize the (premise, hypothesis) pairs
    inputs = tokenizer(
        [text] * len(hypotheses), 
        hypotheses, 
        return_tensors='pt', 
        padding=True,
        truncation=True
    )

    # Run the model
    with torch.no_grad():
        logits = model(**inputs).logits

    # Multi-label or single-label logic
    if multi_label or len(candidate_labels) == 1:
        # Apply softmax over [contradiction, entailment] for each label independently
        entail_contr_logits = logits[:, [0, entailment_id]]
        probs = torch.softmax(entail_contr_logits, dim=1)
        entail_probs = probs[:, 1]
    else:
        # Apply softmax over all labels (single-label setting)
        entail_logits = logits[:, entailment_id]
        entail_probs = torch.softmax(entail_logits, dim=0)

    # Convert to Python list for easy interpretation
    entail_probs = entail_probs.tolist()

    # Plot the probabilities
    if plot:
        plt.figure(figsize=(6, 2))
        plt.barh(candidate_labels, entail_probs, color='skyblue')
        plt.xlabel("Probability")
        plt.title(f"Zero-Shot NLI Classification")
        plt.gca().invert_yaxis()
        plt.grid(alpha=0.3, linestyle='--')
        plt.tight_layout()
        plt.show()

    return {
        "sequence": text,
        "labels": candidate_labels,
        "scores": entail_probs
    }

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "facebook/bart-large-mnli"
tokenizer = AutoTokenizer.from_pretrained(model_name, clean_up_tokenization_spaces=False)
tokenizer.clean_up_tokenization_spaces = False
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
text = "Opposition party gains ground ahead of national election."
labels = ["economics", "sports", "politics"]

ZeroShotNLI(tokenizer, model, text, labels, plot=True)

### Multi-Label Variant

- With `multi_label=False`, the labels are treated as mutually exclusive.
- With `multi_label=True`, each label is scored independently.


In [ ]:
ZeroShotNLI(tokenizer, model, text, labels, multi_label=True)